# Load Packages

In [ ]:
import numpy as np
import pandas as pd
import random
from scipy.sparse import csr_matrix, lil_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Load Dataset

In [ ]:
news_path = '/content/drive/MyDrive/features/news_features/news_features.csv'
news = pd.read_csv(news_path)


#train, val, test set
train_path = '/content/drive/MyDrive/features/user_features/train_user_features.csv'
val_path = '/content/drive/MyDrive/features/user_features/val_user_features.csv'
test_path = '/content/drive/MyDrive/features/user_features/test_user_features.csv'

train = pd.read_csv(train_path)
val = pd.read_csv(val_path)
test = pd.read_csv(test_path)

In [ ]:
train_context_path = '/content/drive/MyDrive/features/context_features/train_context.csv'
train_context = pd.read_csv(train_context_path)
print(train_context.head(10))

   impression_id  user_id                 time  \
0          35263        1  2019-11-09 05:59:43   
1              1        1  2019-11-11 09:05:58   
2          20676        2  2019-11-09 05:28:51   
3           6673        2  2019-11-09 18:21:10   
4          97245        2  2019-11-10 05:43:35   
5         151853        2  2019-11-10 07:13:56   
6         135354        2  2019-11-10 07:16:53   
7          10938        2  2019-11-10 17:37:46   
8         124765        2  2019-11-10 18:02:10   
9          57351        2  2019-11-11 16:50:30   

                                             history  pos_id  \
0  [26659, 7649, 10157, 13853, 11735, 5063, 22437...     515   
1  [26659, 7649, 10157, 13853, 11735, 5063, 22437...   17918   
2  [19437, 27124, 48998, 22350, 37334, 22912, 447...   44926   
3  [19437, 27124, 48998, 22350, 37334, 22912, 447...   49940   
4  [19437, 27124, 48998, 22350, 37334, 22912, 447...    4154   
5  [19437, 27124, 48998, 22350, 37334, 22912, 447...   38476   
6

## Item-Based CF

In [ ]:
#Build user_item_matrix (will transpose later)
def build_user_item_matrix(train_user: pd.DataFrame, train_context: pd.DataFrame):
    """
    Build sparse user-item matrix from two sources:
        1. reading_memory (history) clicks — from train_user
        2. impression clicks (pos_id)      — from train_context

    train_user    : has user_id and reading_memory columns
    train_context : has user_id, pos_id, neg_ids columns
    """
    # expand reading_memory — deduplicate by user first
    memory_exploded = (
        train_user[["user_id", "history"]]
        .drop_duplicates(subset="user_id")
        .explode("history")
        .dropna(subset=["history"])
        .rename(columns={"history": "news_id"})
    )
    memory_exploded["news_id"] = memory_exploded["news_id"].astype(int)

    # impression clicks from train_context
    impression_clicks = train_context[["user_id", "pos_id"]].copy()
    impression_clicks = impression_clicks.rename(columns={"pos_id": "news_id"})
    impression_clicks["news_id"] = impression_clicks["news_id"].astype(int)

    all_user_ids    = train_user["user_id"].unique()
    all_article_ids = pd.concat([
        impression_clicks["news_id"],
        memory_exploded["news_id"]
    ]).unique()

    user_index    = {u: i for i, u in enumerate(all_user_ids)}
    article_index = {a: i for i, a in enumerate(all_article_ids)}

    n_users    = len(user_index)
    n_articles = len(article_index)

    matrix = lil_matrix((n_users, n_articles), dtype=np.float32)

    # source 1 — impression clicks (pos_id from train_context)
    for _, row in impression_clicks.iterrows():
        if row["user_id"] in user_index and row["news_id"] in article_index:
            u = user_index[row["user_id"]]
            a = article_index[row["news_id"]]
            matrix[u, a] = 1.0

    # source 2 — reading memory clicks
    for _, row in memory_exploded.iterrows():
        if row["user_id"] in user_index and row["news_id"] in article_index:
            u = user_index[row["user_id"]]
            a = article_index[row["news_id"]]
            matrix[u, a] = 1.0

    matrix = matrix.tocsr()

    print(f"User-item matrix: {n_users:,} users x {n_articles:,} articles")
    print(f"Non-zero entries: {matrix.nnz:,}")
    print(f"Density: {matrix.nnz / (n_users * n_articles):.6f}")
    return matrix, user_index, article_index


In [ ]:
#Compute item-item cosine similarity and find top-K similar articles
def compute_item_similarity_and_top_k(
    matrix     : csr_matrix,
    k          : int = 20,
    batch_size : int = 500,
):
    """
    Compute item-item cosine similarity and find top-K similar articles in one pass.
    Transposes matrix so articles are rows and users are columns.
    """
    item_matrix   = matrix.T.tocsr()
    n_articles    = item_matrix.shape[0]
    top_k_indices = np.zeros((n_articles, k), dtype=np.int32)
    top_k_sims    = np.zeros((n_articles, k), dtype=np.float32)

    print(f"Computing item-item similarity for {n_articles:,} articles...")

    for start in range(0, n_articles, batch_size):
        end       = min(start + batch_size, n_articles)
        batch_sim = cosine_similarity(item_matrix[start:end], item_matrix)

        # zero self-similarity
        for i in range(end - start):
            batch_sim[i, start + i] = 0.0

        # top-K per row using argpartition
        for i, sim_row in enumerate(batch_sim):
            top_k = np.argpartition(sim_row, -k)[-k:]
            top_k = top_k[np.argsort(-sim_row[top_k])]
            top_k_indices[start + i] = top_k
            top_k_sims[start + i]    = sim_row[top_k]

        if start % 10000 == 0:
            print(f"  {start:,} / {n_articles:,} done")

    print("Item similarity done.")
    return top_k_indices, top_k_sims

In [ ]:
#Score each candidate article based on its similarity to articles in the user's history.
def score_candidates(
    user_memory_idxs : list,
    candidate_idxs   : list,
    top_k_indices    : np.ndarray,
    top_k_sims       : np.ndarray,
) -> np.ndarray:
    """
    Score each candidate article based on its similarity to
    articles in the user's reading memory.

    score(i) = Σ sim(i, j) for j in reading_memory / len(reading_memory)
    """
    if len(user_memory_idxs) == 0:
        return np.zeros(len(candidate_idxs), dtype=np.float32)

    scores     = np.zeros(len(candidate_idxs), dtype=np.float32)
    memory_set = set(user_memory_idxs)

    for c_pos, candidate_idx in enumerate(candidate_idxs):
        neighbour_idxs = top_k_indices[candidate_idx]
        neighbour_sims = top_k_sims[candidate_idx]

        for n_idx, n_sim in zip(neighbour_idxs, neighbour_sims):
            if n_idx in memory_set:
                scores[c_pos] += n_sim

        scores[c_pos] /= len(user_memory_idxs)

    return scores

In [ ]:
#Get article indices for a user's history.
def get_user_memory_idxs(
    user_id        : int,
    user_index     : dict,
    article_index  : dict,
    matrix         : csr_matrix,
    history : list = None,
) -> list:
    """
    Get article indices for a user's reading memory.
    Source 1 — from matrix row (impression clicks + reading memory from training)
    Source 2 — from raw reading_memory list (catches additional articles)
    """
    memory_idxs = []

    if user_id in user_index:
        u_idx       = user_index[user_id]
        memory_idxs = list(matrix[u_idx].nonzero()[1])

    if history is not None:
        for news_id in history:
            try:
                news_id = int(news_id)
                if news_id in article_index and article_index[news_id] not in memory_idxs:
                    memory_idxs.append(article_index[news_id])
            except (ValueError, TypeError):
                continue

    return memory_idxs


In [ ]:
#performance metrics
def compute_mrr(labels: list, scores: list) -> float:
    ranked = [l for _, l in sorted(zip(scores, labels), reverse=True)]
    for rank, label in enumerate(ranked, start=1):
        if label == 1:
            return 1.0 / rank
    return 0.0


def compute_ndcg(labels: list, scores: list, k: int) -> float:
    ranked = [l for _, l in sorted(zip(scores, labels), reverse=True)][:k]
    dcg    = sum((2**l - 1) / np.log2(r + 2) for r, l in enumerate(ranked))
    ideal  = sorted(labels, reverse=True)[:k]
    idcg   = sum((2**l - 1) / np.log2(r + 2) for r, l in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def compute_map(labels: list, scores: list) -> float:
    ranked        = [l for _, l in sorted(zip(scores, labels), reverse=True)]
    num_positives = sum(labels)
    if num_positives == 0:
        return 0.0
    hits, precisions = 0, []
    for rank, label in enumerate(ranked, start=1):
        if label == 1:
            hits += 1
            precisions.append(hits / rank)
    return sum(precisions) / num_positives

In [ ]:
#evaluation
def evaluate(
    eval_df       : pd.DataFrame,
    matrix        : csr_matrix,
    top_k_indices : np.ndarray,
    top_k_sims    : np.ndarray,
    user_index    : dict,
    article_index : dict,
) -> dict:
    aucs, mrrs, ndcg5s, ndcg10s, maps = [], [], [], [], []
    skipped = 0

    for _, row in eval_df.iterrows():

        user_id        = row["user_id"]
        pos_ids        = list(row["pos_ids"])
        neg_ids        = list(row["neg_ids"])
        all_candidates = pos_ids + neg_ids
        labels         = [1] * len(pos_ids) + [0] * len(neg_ids)

        # shuffle to remove position bias
        combined       = list(zip(all_candidates, labels))
        random.shuffle(combined)
        all_candidates, labels = zip(*combined)
        all_candidates = list(all_candidates)
        labels         = list(labels)

        # vocab filter — keep only articles seen in training
        valid_pairs = [
            (n, l) for n, l in zip(all_candidates, labels)
            if n in article_index
        ]

        pos_kept = sum(1 for _, l in valid_pairs if l == 1)
        neg_kept = sum(1 for _, l in valid_pairs if l == 0)

        if pos_kept == 0 or neg_kept == 0:
            skipped += 1
            continue

        valid_news_ids, labels = zip(*valid_pairs)
        labels         = list(labels)
        candidate_idxs = [article_index[n] for n in valid_news_ids]

        # get user reading memory indices
        history     = list(row["history"]) if row["history"] is not None else []
        memory_idxs = get_user_memory_idxs(
            user_id, user_index, article_index, matrix, history
        )

        scores = score_candidates(
            memory_idxs, candidate_idxs, top_k_indices, top_k_sims
        ).tolist()

        if len(set(labels)) > 1:
            aucs.append(roc_auc_score(labels, scores))
        mrrs.append(compute_mrr(labels, scores))
        ndcg5s.append(compute_ndcg(labels, scores, k=5))
        ndcg10s.append(compute_ndcg(labels, scores, k=10))
        maps.append(compute_map(labels, scores))

    print(f"Skipped {skipped:,} impressions (cold-start items or no positives/negatives)")
    return {
        "AUC"    : float(np.mean(aucs))    if aucs    else 0.0,
        "MRR"    : float(np.mean(mrrs))    if mrrs    else 0.0,
        "MAP"    : float(np.mean(maps))    if maps    else 0.0,
        "nDCG@5" : float(np.mean(ndcg5s)) if ndcg5s  else 0.0,
        "nDCG@10": float(np.mean(ndcg10s))if ndcg10s else 0.0,
    }


In [ ]:
#recommend (exclude articles user already read)
def recommend(
    user_id        : int,
    history : list,
    user_index     : dict,
    article_index  : dict,
    top_k_indices  : np.ndarray,
    top_k_sims     : np.ndarray,
    matrix         : csr_matrix,
    top_n          : int = 10,
) -> list:
    """
    Return top-N recommended news_ids for a given user.
    Excludes articles already in the user's reading memory.
    """
    memory_idxs = get_user_memory_idxs(
        user_id, user_index, article_index, matrix, history
    )

    if len(memory_idxs) == 0:
        print(f"User {user_id} has no reading memory — cannot make CF recommendations.")
        return []

    seen_idxs        = set(memory_idxs)
    all_article_idxs = [idx for idx in article_index.values() if idx not in seen_idxs]

    scores      = score_candidates(memory_idxs, all_article_idxs, top_k_indices, top_k_sims)
    idx_to_news = {v: k for k, v in article_index.items()}

    ranked = sorted(
        [(idx_to_news[idx], float(score)) for idx, score in zip(all_article_idxs, scores)],
        key=lambda x: x[1],
        reverse=True,
    )
    return ranked[:top_n]


# Run item-based CF on train val and test

In [ ]:
import ast

# convert history from string to list
train["history"] = train["history"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# also convert for train_context if needed
train_context["history"] = train_context["history"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

for df in [val, test]:
    df["history"] = df["history"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    df["pos_ids"] = df["pos_ids"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    df["neg_ids"] = df["neg_ids"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

In [ ]:
matrix, user_index, article_index = build_user_item_matrix(train, train_context)

User-item matrix: 40,148 users x 35,982 articles
Non-zero entries: 936,005
Density: 0.000648


In [ ]:
K = 200
top_k_indices, top_k_sims = compute_item_similarity_and_top_k(matrix, k=K)

Computing item-item similarity for 35,982 articles...
  0 / 35,982 done
  10,000 / 35,982 done
  20,000 / 35,982 done
  30,000 / 35,982 done
Item similarity done.


In [ ]:
#evaluate on val
val_results = evaluate(val, matrix, top_k_indices, top_k_sims, user_index, article_index)
for metric, value in val_results.items():
    print(f"  {metric}: {value:.4f}")

Skipped 18,149 impressions (cold-start items or no positives/negatives)
  AUC: 0.5704
  MRR: 0.4391
  MAP: 0.4287
  nDCG@5: 0.4499
  nDCG@10: 0.5119


In [ ]:
#evaluate on test
test_results = evaluate(test, matrix, top_k_indices, top_k_sims, user_index, article_index)
for metric, value in test_results.items():
    print(f"  {metric}: {value:.4f}")

Skipped 26,315 impressions (cold-start items or no positives/negatives)
  AUC: 0.5760
  MRR: 0.5789
  MAP: 0.5740
  nDCG@5: 0.6205
  nDCG@10: 0.6584


In [ ]:
#recommend to user
example_user   = int(train["user_id"].iloc[0])
example_memory = list(train[train["user_id"] == example_user]["history"].iloc[0])
print(f"\nTop-10 recommendations for user {example_user}:")
recs = recommend(
    example_user, example_memory,
    user_index, article_index,
    top_k_indices, top_k_sims, matrix
)
for news_id, score in recs:
    print(f"  news_id={news_id}  score={score:.4f}")



Top-10 recommendations for user 1:
  news_id=11383  score=0.0771
  news_id=34585  score=0.0690
  news_id=21392  score=0.0678
  news_id=23815  score=0.0675
  news_id=22147  score=0.0616
  news_id=16081  score=0.0584
  news_id=37947  score=0.0538
  news_id=3785  score=0.0525
  news_id=107  score=0.0520
  news_id=7359  score=0.0520
